# Policy Agent — Team Walkthrough

This notebook walks your team through **building, deploying, and testing** the policy agent end-to-end:

1. Environment check (gcloud, ADC, Python)
2. Clone repo and install dependencies
3. Configure project / region / buckets
4. Create GCS buckets and seed sample policies
5. Deploy to Vertex AI Agent Engine with **Agent Identity** (runs `deploy.py`)
6. Capture Reasoning Engine resource name
7. Register as a Gemini Enterprise add-on agent (runs `register_gemini_enterprise.py`)
8. Test the agent in Gemini Enterprise — **expected to fail** (missing permissions)
9. Grant the agent principal `roles/storage.objectViewer` on the policy bucket
10. Re-test with new permissions — should succeed
11. Teardown (stop billing when you're done)



## 1. Environment check

Verify the prerequisites before you do anything that costs money.

**Common gotcha:** the `gcloud` CLI active account and your Application Default Credentials (ADC) can be *different* accounts. The Python SDK uses ADC, so if they differ you'll see `403 storage.buckets.get` during deploy. The cell below prints both — they should match (or at least both have access to the target project).

In [ ]:
import shutil, subprocess, sys

print('Python :', sys.version.split()[0])
for tool in ('gcloud', 'gsutil'):
    path = shutil.which(tool)
    print(f'{tool:7}: {path or "NOT FOUND — install the Google Cloud SDK"}')

def sh(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except subprocess.CalledProcessError as e:
        return f'ERROR: {e.output.strip()}'

print('\ngcloud active account :', sh('gcloud config get-value account'))
print('gcloud active project :', sh('gcloud config get-value project'))
print('\nADC identity (this is what the Python SDK will use):')
print(sh('gcloud auth application-default print-access-token | head -c 1 > /dev/null && '
        'curl -s -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" '
        'https://openidconnect.googleapis.com/v1/userinfo'))

If ADC is missing or wrong, run this in a terminal (opens a browser), then restart the kernel:
```
gcloud auth application-default login --account=<your-account>
```

## 2. Clone repo and install dependencies

Installs the package in editable mode so code edits in `policy_agent/` are picked up without reinstalling.

> **Note:** If you are running this notebook from **inside** the already-cloned repo , the clone step is automatically skipped and the cell just does a `git pull` + `pip install`. No action needed.

In [ ]:
import subprocess, sys, os

REPO_URL = 'https://github.com/avnit/agent-identity.git'
# Repo root = parent of the notebooks/ directory
# REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
REPO_DIR = '/content/agent-identity'  # @param {type:"string"}
os.environ['REPO_DIR'] = REPO_DIR
print(f'Repo root: {REPO_DIR}')
REPO_BASE = '/content'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    # Already inside the cloned repo -- just pull latest
    print('Repo already present -- pulling latest changes...')
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull'],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr or '(already up to date)')
else:
    # Fresh environment (e.g. Colab) -- clone the repo
    print(f'Cloning {REPO_URL} into {REPO_DIR} ...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_BASE],
        capture_output=True, text=True
    )
    # if result.returncode != 0:
    #    raise RuntimeError(f'git clone failed:\n{result.stderr}')
    print(result.stdout or 'Clone complete')

# Install in editable mode from repo root
print('\nInstalling package...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
print('\u2705 Install complete')

## 3. Configuration

**Edit this cell** — everything below reads from these variables. These match the env vars expected by `deploy.py` and `register_gemini_enterprise.py`.

| Variable | Required by | Notes |
|---|---|---|
| `PROJECT_ID` | deploy.py | Your GCP project ID |
| `LOCATION` | deploy.py | Vertex AI region (e.g. `us-central1`) |
| `STAGING_BUCKET` | deploy.py | Agent Engine staging bucket (must be globally unique) |
| `POLICY_BUCKET` | deploy.py | Bucket holding policy documents (globally unique) |
| `POLICY_PREFIX` | deploy.py | Object prefix for policies (default `policies/`) |
| `DISPLAY_NAME` | deploy.py | Name shown in Agent Engine console |
| `AGENT_MODEL` | policy_agent | Gemini model to use |
| `GEMINI_ENTERPRISE_APP_ID` | register_gemini_enterprise.py | Your GE app ID |
| `GEMINI_ENTERPRISE_LOCATION` | register_gemini_enterprise.py | `global`, `us`, or `eu` |

In [ ]:
import os

# ── REQUIRED — edit these ──────────────────────────────────────────────────
PROJECT_ID             = 'your-project-id'  # @param {type:"string"}
LOCATION               = 'us-central1'       # @param {type:"string"}
STAGING_BUCKET         = ''                  # @param {type:"string"}
POLICY_BUCKET          = ''                  # @param {type:"string"}
POLICY_PREFIX          = 'policies/'         # @param {type:"string"}
DISPLAY_NAME           = 'policy-agent'      # @param {type:"string"}
AGENT_MODEL            = 'gemini-2.5-flash'  # @param {type:"string"}

# ── REQUIRED for Step 6 — edit these ────────────────────────────────────────
GEMINI_ENTERPRISE_APP_ID   = ''       # @param {type:"string"}
GEMINI_ENTERPRISE_LOCATION = 'global'  # @param {type:"string"}

# ── Filled automatically after Step 5 deploy (or paste manually) ─────────────
REASONING_ENGINE = ''  # @param {type:"string"}

# ── Auto-derive bucket names if left blank ───────────────────────────────────
if not STAGING_BUCKET:
    STAGING_BUCKET = f'{PROJECT_ID}-agent-staging'
if not POLICY_BUCKET:
    POLICY_BUCKET = f'{PROJECT_ID}-policies'

# ── Validate ─────────────────────────────────────────────────────────────────
assert PROJECT_ID and PROJECT_ID != 'your-project-id', \
    '❌ Set PROJECT_ID to your actual GCP project ID before proceeding'

# ── Propagate to subprocess env (deploy.py / register_gemini_enterprise.py) ──
os.environ['GOOGLE_CLOUD_PROJECT']       = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION']      = LOCATION
os.environ['STAGING_BUCKET']             = STAGING_BUCKET
os.environ['POLICY_BUCKET']              = POLICY_BUCKET
os.environ['POLICY_PREFIX']              = POLICY_PREFIX
os.environ['DISPLAY_NAME']               = DISPLAY_NAME
os.environ['AGENT_MODEL']                = AGENT_MODEL
os.environ['GEMINI_ENTERPRISE_APP_ID']   = GEMINI_ENTERPRISE_APP_ID
os.environ['GEMINI_ENTERPRISE_LOCATION'] = GEMINI_ENTERPRISE_LOCATION
if REASONING_ENGINE:
    os.environ['REASONING_ENGINE'] = REASONING_ENGINE

print('Configuration:')
for k in ('PROJECT_ID','LOCATION','STAGING_BUCKET','POLICY_BUCKET',
          'POLICY_PREFIX','DISPLAY_NAME','AGENT_MODEL',
          'GEMINI_ENTERPRISE_APP_ID','GEMINI_ENTERPRISE_LOCATION',
          'REASONING_ENGINE'):
    print(f'  {k:30} = {eval(k)}')
print('\n✅ Configuration set')

## 4. Enable APIs, create buckets, seed sample policies

Idempotent — re-runs no-op if the API is already enabled or the bucket already exists.

In [ ]:
import subprocess

def run(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0 and 'already exists' not in (r.stderr or '').lower():
        print('STDERR:', r.stderr.strip())
    return r

run(f'gcloud services enable aiplatform.googleapis.com storage.googleapis.com discoveryengine.googleapis.com --project={PROJECT_ID}')
run(f'gcloud storage buckets create gs://{STAGING_BUCKET} --project={PROJECT_ID} --location={LOCATION} --uniform-bucket-level-access')
run(f'gcloud storage buckets create gs://{POLICY_BUCKET}  --project={PROJECT_ID} --location={LOCATION} --uniform-bucket-level-access')
run(f'gcloud storage cp {REPO_DIR}/sample_policies/*.md gs://{POLICY_BUCKET}/{POLICY_PREFIX} --project={PROJECT_ID}')
run(f'gcloud storage ls gs://{POLICY_BUCKET}/{POLICY_PREFIX} --project={PROJECT_ID}')

## 5. Deploy to Vertex AI Agent Engine with Agent Identity

**This step creates billable resources** — ~5–15 minutes while the container builds.

This runs `deploy.py` from the repo root. The key config in that script is `identity_type=AGENT_IDENTITY`, which makes Agent Engine mint a per-agent SPIFFE principal you can grant IAM to directly.

The script prints the `reasoningEngine` resource name at the end — it's captured automatically into `REASONING_ENGINE` below.

In [ ]:
import subprocess, sys, os, re

# REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
deploy_script = os.path.join(REPO_DIR, 'deploy.py')

print(f'Running: python {deploy_script}')
print('This takes 5–15 minutes — output streams below...\n')

result = subprocess.run(
    [sys.executable, deploy_script],
    capture_output=True,   # stream stdout/stderr live
    text=True,
    cwd=REPO_DIR,
    env=os.environ.copy(),
)

if result.returncode != 0:
    raise RuntimeError(f'deploy.py exited with code {result.returncode}')

print('\n✅ Deployment complete')
print('\n⚠️  Copy the reasoningEngine value printed above and paste it into'
      ' REASONING_ENGINE in the next cell.')

## 6. Capture Reasoning Engine resource name

After `deploy.py` completes, it prints the `reasoningEngine` resource path.
Paste that value below — it's required by the register, grant-IAM, and remote-test steps.

> **Tip:** The value looks like:
> `projects/123456789/locations/us-central1/reasoningEngines/987654321098765432`

In [ ]:
# ── Paste the reasoningEngine output from deploy.py here ─────────────────────
REASONING_ENGINE = ''  # e.g. 'projects/123456/locations/us-central1/reasoningEngines/987654321'

assert REASONING_ENGINE, '❌ Set REASONING_ENGINE to the value printed by deploy.py'
os.environ['REASONING_ENGINE'] = REASONING_ENGINE
print(f'REASONING_ENGINE = {REASONING_ENGINE}')

## 7. Register as a Gemini Enterprise add-on agent

Runs `register_gemini_enterprise.py` from the repo root. This POSTs to the Discovery Engine Assistants API to register the deployed reasoning engine as an add-on agent in your Gemini Enterprise app.

**Prerequisites:**
- `GEMINI_ENTERPRISE_APP_ID` must be set in Step 3
- `REASONING_ENGINE` must be set from Step 5
- A Gemini Enterprise app must already exist in this project

In [ ]:
import subprocess, sys, os

assert GEMINI_ENTERPRISE_APP_ID, \
    '❌ Set GEMINI_ENTERPRISE_APP_ID in Step 3 before running this cell'
assert REASONING_ENGINE, \
    '❌ Set REASONING_ENGINE in Step 6 before running this cell'

# REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
register_script = os.path.join(REPO_DIR, 'register_gemini_enterprise.py')

print(f'Running: python {register_script}')
print(f'  GEMINI_ENTERPRISE_APP_ID = {GEMINI_ENTERPRISE_APP_ID}')
print(f'  REASONING_ENGINE         = {REASONING_ENGINE}\n')

result = subprocess.run(
    [sys.executable, register_script],
    capture_output=False,
    text=True,
    cwd=REPO_DIR,
    env=os.environ.copy(),
)

if result.returncode != 0:
    raise RuntimeError(f'register_gemini_enterprise.py exited with code {result.returncode}')

print('\n✅ Agent registered with Gemini Enterprise')

## 8. Test in Gemini Enterprise — ⚠️ Expected to fail

Go to the **Gemini Enterprise** app in the Google Cloud console and ask the Policy Assistant agent a question, e.g.:

> *"What is my work from home policy?"*

**This will fail** with a permission error — the agent principal has not yet been granted access to the policy bucket. You will see an error like:

```
403 storage.objects.list denied on gs://<POLICY_BUCKET>
```

This is expected and demonstrates why Agent Identity + IAM binding matters. Continue to Step 8 to fix it.

---

**Where to find the app:**
1. Go to [console.cloud.google.com](https://console.cloud.google.com)
2. Navigate to **Gemini Enterprise** (or Agentspace)
3. Open your app and select the **Policy Assistant** agent
4. Send a test question and observe the error

## 9. Grant the agent principal access to GCS

Fetches the reasoning engine's `effectiveIdentity` (the per-agent SPIFFE principal) and binds `roles/storage.objectViewer` on the policy bucket to it.

This is the fix for the failure in Step 7 — no service-account key needed.

In [ ]:
import json, subprocess

assert REASONING_ENGINE, '❌ Set REASONING_ENGINE in Step 6 before running this cell'

# Fetch the reasoning engine resource to get the effectiveIdentity
token = subprocess.check_output(
    'gcloud auth application-default print-access-token', shell=True, text=True
).strip()

resp = subprocess.check_output(
    f'curl -s -H "Authorization: Bearer {token}" '
    f'https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}',
    shell=True, text=True,
)
engine = json.loads(resp)
effective_identity = engine['spec']['effectiveIdentity']
AGENT_PRINCIPAL = f'principal://{effective_identity}'
print(f'Agent principal: {AGENT_PRINCIPAL}\n')

# Bind roles/storage.objectViewer on the policy bucket
run(
    f'gcloud storage buckets add-iam-policy-binding gs://{POLICY_BUCKET} '
    f'--member="{AGENT_PRINCIPAL}" '
    f'--role="roles/storage.objectViewer" '
    f'--project={PROJECT_ID}'
)
print('\n✅ IAM binding applied — allow ~30 seconds for propagation')

## 10. Re-test in Gemini Enterprise — ✅ Should succeed

Now that the agent principal has `roles/storage.objectViewer` on the policy bucket, go back to the Gemini Enterprise app and ask the same question again:

> *"What is my work from home policy?"*

The agent should now successfully:
1. Call `list_policies` → `search_policies` → `get_policy_document`
2. Read from `gs://<POLICY_BUCKET>/policies/` via its **Agent Identity** principal (no service-account key)
3. Return a cited answer from the policy documents

You can also run a quick remote test from the notebook to confirm:

In [ ]:
import vertexai

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=dict(api_version='v1beta1'),
)

remote = client.agent_engines.get(name=REASONING_ENGINE)

async def ask_remote(prompt: str) -> str:
    session = await remote.async_create_session(user_id='nb-user')
    out = []
    async for event in remote.async_stream_query(
        user_id='nb-user', session_id=session['id'], message=prompt,
    ):
        parts = (event.get('content') or {}).get('parts', [])
        for p in parts:
            if 'text' in p:
                out.append(p['text'])
    return ''.join(out)

print('Testing deployed agent with new permissions...\n')
answer = await ask_remote('What is my work from home policy?')
print(answer)
print('\n---')
print(await ask_remote('What is my work from home policy?'))


## 11. Teardown (stop ongoing billing)

Deletes the reasoning engine. The buckets stay (empty buckets cost ~nothing) — the stale IAM binding on the policy bucket becomes a no-op once the principal is gone, but you can remove it if you like.

In [ ]:
import subprocess
token = subprocess.check_output('gcloud auth application-default print-access-token', shell=True, text=True).strip()
print(subprocess.check_output(
    f'curl -s -X DELETE -H "Authorization: Bearer {token}" '
    f'"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}?force=true"',
    shell=True, text=True,
))

In [ ]:
# Optional: remove the now-stale IAM binding
run(
    f'gcloud storage buckets remove-iam-policy-binding gs://{POLICY_BUCKET} '
    f'--member="{AGENT_PRINCIPAL}" '
    f'--role="roles/storage.objectViewer" '
    f'--project={PROJECT_ID}'
)

## Troubleshooting

| Error you see | Fix |
|---|---|
| `403 storage.buckets.get` during deploy | ADC identity ≠ gcloud identity. Re-run `gcloud auth application-default login --account=<right-account>` and restart the kernel. |
| `400 Environment variable name 'GOOGLE_CLOUD_PROJECT' is reserved` | Don't put `GOOGLE_CLOUD_PROJECT` / `GOOGLE_CLOUD_LOCATION` in `env_vars` — Agent Engine injects them. |
| `ModuleNotFoundError: No module named 'policy_agent'` in engine logs | `extra_packages=['policy_agent']` must be set so the local source ships to the container. |
| `403 storage.objects.list denied` at query time | Step 8 didn't run or was run against the wrong bucket. Re-check `POLICY_BUCKET` and the agent principal. |
| `Missing required env var: STAGING_BUCKET` | Make sure Step 3 ran and `os.environ` was populated before running Step 5. |
| Deploy succeeds but engine won't start | `gcloud logging read 'resource.type="aiplatform.googleapis.com/ReasoningEngine" AND resource.labels.reasoning_engine_id="<ID>"' --project=<PROJECT> --limit=50 --freshness=30m` |